# Direct generation check for a few utility names

This small diagnostic uses PUDL's annual EIA-923 generation-and-fuel table.
It does **not** combine parent companies, allocate joint ownership, or rename utilities.
The purpose is to confirm the exact utility names and see the raw generation breakdown reported under each name.

**Years:** 2021–2024  
**Measure:** actual net generation in MWh  
**Interpretation:** generation associated with the reporting plant utility/operator, not the full power mix delivered to retail customers.


In [ ]:
# Setup
!pip -q install --upgrade duckdb

import duckdb
import pandas as pd

PUDL_FILE = (
    "https://s3.us-west-2.amazonaws.com/"
    "pudl.catalyst.coop/stable/"
    "out_eia923__yearly_generation_fuel_combined.parquet"
)

con = duckdb.connect()
try:
    con.execute("LOAD httpfs")
except duckdb.Error:
    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")

print("Ready.")


In [ ]:
# Step 1: show every exact PUDL utility name matching a few simple search terms
matches = con.execute(
    f"""
    SELECT DISTINCT
        utility_id_eia,
        utility_name_eia
    FROM read_parquet('{PUDL_FILE}')
    WHERE EXTRACT(YEAR FROM report_date) BETWEEN 2021 AND 2024
      AND utility_name_eia IS NOT NULL
      AND (
          LOWER(utility_name_eia) LIKE '%new york state elec%'
          OR LOWER(utility_name_eia) LIKE '%nstar electric%'
          OR LOWER(utility_name_eia) LIKE '%consolidated edison%'
          OR LOWER(utility_name_eia) LIKE '%united illuminating%'
      )
    ORDER BY utility_name_eia, utility_id_eia
    """
).df()

display(matches)


In [ ]:
# Step 2: retrieve the raw annual generation breakdown for those matched names
breakdown = con.execute(
    f"""
    SELECT
        CAST(EXTRACT(YEAR FROM report_date) AS INTEGER) AS year,
        utility_id_eia,
        utility_name_eia,
        COALESCE(fuel_type_code_pudl, energy_source_code, 'unknown') AS fuel,
        SUM(net_generation_mwh) AS net_generation_mwh
    FROM read_parquet('{PUDL_FILE}')
    WHERE EXTRACT(YEAR FROM report_date) BETWEEN 2021 AND 2024
      AND utility_name_eia IS NOT NULL
      AND (
          LOWER(utility_name_eia) LIKE '%new york state elec%'
          OR LOWER(utility_name_eia) LIKE '%nstar electric%'
          OR LOWER(utility_name_eia) LIKE '%consolidated edison%'
          OR LOWER(utility_name_eia) LIKE '%united illuminating%'
      )
    GROUP BY 1, 2, 3, 4
    ORDER BY utility_name_eia, year, net_generation_mwh DESC
    """
).df()

if breakdown.empty:
    raise ValueError("No generation records matched the search terms.")

breakdown['fuel'] = (
    breakdown['fuel']
    .astype(str)
    .str.replace('_', ' ', regex=False)
    .str.title()
)

# Calculate shares without merging or renaming any utility.
positive = breakdown[breakdown['net_generation_mwh'] > 0].copy()
positive['total_positive_generation_mwh'] = (
    positive.groupby(['year', 'utility_id_eia'])['net_generation_mwh']
    .transform('sum')
)
positive['generation_share_percent'] = (
    100 * positive['net_generation_mwh']
    / positive['total_positive_generation_mwh']
)

display(
    positive[
        [
            'year', 'utility_id_eia', 'utility_name_eia', 'fuel',
            'net_generation_mwh', 'generation_share_percent'
        ]
    ].style.format({
        'net_generation_mwh': '{:,.0f}',
        'generation_share_percent': '{:.1f}%'
    })
)

# Compact latest-year comparison.
latest = positive[positive['year'] == 2024].pivot_table(
    index=['utility_id_eia', 'utility_name_eia'],
    columns='fuel',
    values='generation_share_percent',
    aggfunc='sum',
    fill_value=0,
)

display(latest.style.format('{:.1f}%').set_caption('2024 direct generation mix'))
